In [22]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, RobustScaler
from sklearn.model_selection import train_test_split
import joblib
import pickle

import warnings
warnings.filterwarnings('ignore')

In [1]:
# Installer les outils nécessaires
!pip install --upgrade google-cloud-storage


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 174.9/174.9 kB 4.2 MB/s eta 0:00:00
  Attempting uninstall: google-cloud-storage
    Found existing installation: google-cloud-storage 2.19.0
    Uninstalling google-cloud-storage-2.19.0:
      Successfully uninstalled google-cloud-storage-2.19.0


In [2]:
# Authentification Google
from google.colab import auth
auth.authenticate_user()

In [3]:
# Bibliothèques
from google.cloud import storage
import pandas as pd

In [4]:
# Créer le client de Google Cloud Storage
client = storage.Client()

In [5]:
# Définir le bucket et le nom du fichier
bucket_name = 'mon-bucket-22go'  # Remplace si ton bucket a un nom différent
blob_name = 'ddos.csv'
local_path = '/content/ddos.csv'


In [6]:
# Télécharger le fichier
bucket = client.bucket(bucket_name)
blob = bucket.blob(blob_name)
blob.download_to_filename(local_path)

In [7]:
print("✅ Fichier téléchargé localement dans Colab")

✅ Fichier téléchargé localement dans Colab


In [8]:
# Charger le fichier dans pandas
df = pd.read_csv(local_path)
print("✅ Chargement terminé")

# Afficher un aperçu
df.head()

<ipython-input-8-5a86c4cb7320>:2: DtypeWarning: Columns (85) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(local_path)


✅ Chargement terminé


,Unnamed: 0,Flow ID,Source IP,Source Port,Destination IP,Destination Port,Protocol,Timestamp,Flow Duration,Total Fwd Packets,...,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,SimillarHTTP,Inbound,Label
0,0,172.16.0.5-192.168.50.1-60675-80-6,172.16.0.5,60675,192.168.50.1,80,6,2018-12-01 09:17:11.183810,5220876,12,...,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,205.174.165.72/c.php,1,DrDoS_NTP
1,7,172.16.0.5-192.168.50.1-60676-80-6,172.16.0.5,60676,192.168.50.1,80,6,2018-12-01 09:17:11.205636,12644252,5,...,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0,1,DrDoS_NTP
2,12858,192.168.50.7-65.55.163.78-50458-443-6,65.55.163.78,443,192.168.50.7,50458,6,2018-12-01 09:17:12.634569,3,2,...,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0,1,BENIGN
3,10191,192.168.50.7-65.55.163.78-50465-443-6,65.55.163.78,443,192.168.50.7,50465,6,2018-12-01 09:17:13.458370,3,2,...,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0,1,BENIGN
4,239,192.168.50.253-224.0.0.5-0-0-0,192.168.50.253,0,224.0.0.5,0,0,2018-12-01 09:17:13.470913,114329232,52,...,2.466441,15.0,6.0,9527428.0,248706.681286,9950741.0,9092248.0,0,0,BENIGN


In [10]:
df.shape

(50063112, 88)

In [11]:
# Supprimer les espaces avant/après les noms de colonnes
df.columns = df.columns.str.strip()

# Refaire un print propre
print(df.columns.tolist())

['Unnamed: 0', 'Flow ID', 'Source IP', 'Source Port', 'Destination IP', 'Destination Port', 'Protocol', 'Timestamp', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag

In [13]:
# Afficher les valeurs uniques et leur nombre
print(df["Label"].value_counts())

Label
TFTP             20082580
DrDoS_SNMP        5159870
DrDoS_DNS         5071011
DrDoS_MSSQL       4522492
DrDoS_NetBIOS     4093279
DrDoS_UDP         3134645
DrDoS_SSDP        2610611
DrDoS_LDAP        2179930
Syn               1582289
DrDoS_NTP         1202642
UDP-lag            366461
BENIGN              56863
WebDDoS               439
Name: count, dtype: int64


In [14]:
# lignes avec des valeurs manquantes
# Compter le nombre de lignes avec des valeurs manquantes
nb_lignes_manquantes = df.isnull().any(axis=1).sum()
print(f"Nombre total de lignes avec des valeurs manquantes : {nb_lignes_manquantes}")


Nombre total de lignes avec des valeurs manquantes : 249024


In [15]:
# Suppressions de lignes avec des valeurs manquantes directement du dataframe
# 1. Identifier les lignes à supprimer :
# - Lignes avec NaN (sauf si Label == 'BENIGN')
mask_a_supprimer = df.isnull().any(axis=1) & (df['Label'] != 'BENIGN')
# 2. Supprimer ces lignes du DataFrame :
data = df[~mask_a_supprimer]  # Garder les lignes qui NE correspondent PAS au masque

In [16]:
# 3. Vérification :
print(f"Lignes supprimées : {mask_a_supprimer.sum()}")
print(f"Taille du DataFrame après suppression : {len(data)}")

Lignes supprimées : 248911
Taille du DataFrame après suppression : 49814201


In [17]:
#vérification du label
print(data["Label"].value_counts())

Label
TFTP             20072108
DrDoS_SNMP        5159863
DrDoS_DNS         5071002
DrDoS_MSSQL       4522489
DrDoS_NetBIOS     4093273
DrDoS_UDP         3134643
DrDoS_SSDP        2610610
DrDoS_LDAP        2179928
Syn               1380015
DrDoS_NTP         1202639
UDP-lag            330329
BENIGN              56863
WebDDoS               439
Name: count, dtype: int64


In [18]:
#colonnes avec des valeurs manquantes
# Calculer les valeurs manquantes par colonne
missing_values = data.isnull().sum()
missing_percent = (missing_values / len(data)) * 100

# Filtrer et afficher uniquement les colonnes avec des valeurs manquantes
missing_data = pd.DataFrame({
    'Valeurs manquantes': missing_values,
    'Pourcentage (%)': missing_percent
}).loc[missing_values > 0]  # Ne garder que les colonnes avec NaN

missing_data.sort_values(by='Valeurs manquantes', ascending=False)

,Valeurs manquantes,Pourcentage (%)
Flow Bytes/s,113,0.000227


In [19]:
#Suppression des colonnes avec VM
# Récupérer les noms des colonnes à supprimer (celles avec NaN)
colonnes_a_supprimer = missing_data.index.tolist()

# Supprimer ces colonnes du DataFrame principal
data = data.drop(columns=colonnes_a_supprimer)

# Vérification
print(f"Colonnes supprimées : {colonnes_a_supprimer}")
print(f"Nouvelles dimensions du DataFrame : {data.shape}")

Colonnes supprimées : ['Flow Bytes/s']
Nouvelles dimensions du DataFrame : (49814201, 87)


In [20]:
data.shape

(49814201, 87)

In [23]:
# Valeurs NaN ou infinis
#lignes
print(f"Lignes avec NaN : {data.isna().any(axis=1).sum()}")
print(f"Lignes avec infinis : {(data == np.inf).any(axis=1).sum() + (data == -np.inf).any(axis=1).sum()}")

Lignes avec NaN : 0
Lignes avec infinis : 1114325


In [24]:
# Créer un masque pour identifier les lignes à supprimer :
# 1. Lignes avec NaN ou infinis
mask_problemes = data.isna().any(axis=1) | (data == np.inf).any(axis=1) | (data == -np.inf).any(axis=1)

# 2. ET dont le Label n'est pas 'BENIGN' (à conserver)
mask_a_supprimer = mask_problemes & (data['Label'] != 'BENIGN')

In [25]:
# Supprimer les lignes indésirables
data = data[~mask_a_supprimer]  # Garde les lignes qui ne correspondent PAS au masque

# Vérification
print(f"Lignes supprimées : {mask_a_supprimer.sum()}")
print(f"Taille du DataFrame après nettoyage : {len(data)}")

Lignes supprimées : 1113887
Taille du DataFrame après nettoyage : 48700314


In [26]:
#vérification du label
print(data["Label"].value_counts())

Label
TFTP             19515971
DrDoS_SNMP        5149261
DrDoS_DNS         4908665
DrDoS_MSSQL       4396046
DrDoS_NetBIOS     3963446
DrDoS_UDP         3094002
DrDoS_SSDP        2568569
DrDoS_LDAP        2141300
Syn               1379983
DrDoS_NTP         1195690
UDP-lag            330079
BENIGN              56863
WebDDoS               439
Name: count, dtype: int64


In [27]:
# colonnes
import numpy as np

# Détecter NaN et infinis
has_nan = data.isna().any()
has_inf = (data == np.inf).any() | (data == -np.inf).any()

# Colonnes problématiques
problem_cols = data.columns[has_nan | has_inf]

print("Colonnes avec NaN ou infinis :", problem_cols.tolist())

Colonnes avec NaN ou infinis : ['Flow Packets/s']


In [28]:
# DataFrame récapitulatif
missing_inf_data = pd.DataFrame({
    'Valeurs_manquantes (NaN)': data.isna().sum()[problem_cols],
    '% NaN': (data.isna().mean()[problem_cols] * 100).round(2),
    'Valeurs_infinies (inf)': ((data == np.inf) | (data == -np.inf)).sum()[problem_cols],
    '% inf': ((data == np.inf).mean()[problem_cols] * 100 + (data == -np.inf).mean()[problem_cols] * 100).round(2)
})

missing_inf_data.sort_values(by='Valeurs_manquantes (NaN)', ascending=False)

,Valeurs_manquantes (NaN),% NaN,Valeurs_infinies (inf),% inf
Flow Packets/s,0,0.0,438,0.0


In [29]:
#Suppression
# 1. Récupérer les noms des colonnes à supprimer (index de missing_inf_data)
colonnes_a_supprimer = missing_inf_data.index.tolist()


In [30]:
# 2. Supprimer ces colonnes du DataFrame original (méthode non destructive)
data= data.drop(columns=colonnes_a_supprimer)
# 3. Vérification
print("Colonnes supprimées :", colonnes_a_supprimer)
print("Nouvelles dimensions du DataFrame :", data.shape)

Colonnes supprimées : ['Flow Packets/s']
Nouvelles dimensions du DataFrame : (48700314, 86)


In [31]:
#vérification finale
print(f"Lignes avec NaN : {data.isna().any(axis=1).sum()}")
print(f"Lignes avec infinis : {(data == np.inf).any(axis=1).sum() + (data == -np.inf).any(axis=1).sum()}")

Lignes avec NaN : 0
Lignes avec infinis : 0


In [32]:
data.shape

(48700314, 86)

In [33]:
# Colonnes 100% avec des zéros (Vides)
# Colonnes où TOUTES les valeurs sont 0 (ou 0.0)
colonnes_zero = [col for col in data.columns if (data[col] == 0).all()]

# Afficher les colonnes concernées
print("Colonnes avec uniquement des 0 / 0.0 :", colonnes_zero)

Colonnes avec uniquement des 0 / 0.0 : ['Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'FIN Flag Count', 'PSH Flag Count', 'ECE Flag Count', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate']


In [34]:
# Supprimer ces colonnes du DataFrame (modification directe)
data.drop(columns=colonnes_zero, inplace=True)

In [35]:
# Vérification
print("Nouvelles dimensions du DataFrame :", data.shape)

Nouvelles dimensions du DataFrame : (48700314, 74)


# Colonnes Quasi-Vides

In [36]:
seuil = 0.9

# Colonnes avec ≥90% de zéros (0 ou 0.0)
colonnes_zero_90 = [
    col for col in data.columns
    if (data[col] == 0).mean() >= seuil
]

# Afficher les colonnes concernées
print(f"Colonnes avec ≥{seuil:.0%} de zéros :", colonnes_zero_90)


Colonnes avec ≥90% de zéros : ['Total Backward Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd Header Length', 'Bwd Packets/s', 'Packet Length Std', 'Packet Length Variance', 'SYN Flag Count', 'RST Flag Count', 'ACK Flag Count', 'URG Flag Count', 'CWE Flag Count', 'Down/Up Ratio', 'Avg Bwd Segment Size', 'Subflow Bwd Packets', 'Subflow Bwd Bytes', 'Active Mean', 'Active Std', 'Active Max', 'Active Min', 'Idle Mean', 'Idle Std', 'Idle Max', 'Idle Min', 'SimillarHTTP']


In [37]:
# Supprimer ces colonnes (modification directe)
data.drop(columns=colonnes_zero_90, inplace=True)

In [38]:
# Vérification
print(f"\nDimensions après suppression : {data.shape}")


Dimensions après suppression : (48700314, 39)


# Colonnes Constante

In [39]:
# 1. Détection des colonnes constantes
colonnes_constantes = [col for col in data.columns if data[col].nunique(dropna=False) == 1]

# 2. Affichage des résultats
print("Colonnes constantes détectées :", colonnes_constantes)
print(f"Nombre de colonnes constantes : {len(colonnes_constantes)}")


Colonnes constantes détectées : []
Nombre de colonnes constantes : 0


In [40]:
# 3. Suppression
data.drop(columns=colonnes_constantes, inplace=True)

In [41]:
# 4. Vérification
print("Dimensions après suppression :", data.shape)

Dimensions après suppression : (48700314, 39)


# Lignes Constantes

In [42]:
# 1. Détection des lignes constantes
lignes_constantes = data.apply(lambda row: row.nunique(dropna=False) == 1, axis=1)

# 2. Affichage des résultats
print(f"Lignes constantes détectées : {lignes_constantes.sum()}")
print(f"Exemples d'index : {data[lignes_constantes].index[:5].tolist()}")

Lignes constantes détectées : 0
Exemples d'index : []


In [43]:
# 3. Suppression
data.drop(index=data[lignes_constantes].index, inplace=True)
# 4. Vérification
print("Dimensions après suppression :", data.shape)

Dimensions après suppression : (48700314, 39)


# Lignes redondantes

In [44]:
# 1. Détection des doublons (toutes colonnes)
duplicates_mask = data.duplicated(keep='first')  # Garde la première occurrence
redondantes = data[duplicates_mask]

# 2. Affichage des statistiques
print(f"Nombre de lignes redondantes : {len(redondantes)}")
print(f"Pourcentage de redondance : {len(redondantes)/len(data):.2%}")


Nombre de lignes redondantes : 0
Pourcentage de redondance : 0.00%


In [45]:
# 3. Suppression des doublons
data.drop_duplicates(keep='first', inplace=True)

In [46]:
# 4. Vérification
print(f"\nDimensions : {data.shape}")


Dimensions : (48700314, 39)


# Colonnes Redondantes

In [47]:
# 1. Trouver les colonnes redondantes
cols = data.columns
duplicate_cols = []

for i in range(len(cols)):
    for j in range(i+1, len(cols)):
        if data[cols[i]].equals(data[cols[j]]):
            duplicate_cols.append(cols[j])

In [48]:
# 2. Supprimer les doublons (en gardant une occurrence)
data = data.drop(columns=duplicate_cols)


In [49]:
# 3. Affichage des résultats
print(f"Colonnes redondantes détectées : {duplicate_cols}")
print(f"Shape avant/apres : {data.shape}")

Colonnes redondantes détectées : ['Subflow Fwd Packets', 'Fwd Header Length.1']
Shape avant/apres : (48700314, 37)


# Colonnes quasi-constantes

In [50]:
# Paramètre : seuil à 90%
seuil = 0.9

# Colonne à toujours conserver (même si quasi-constante)
colonne_protegee = 'Label'  # À adapter selon le nom exact de votre colonne

# 1. Détection des colonnes quasi-constantes (en excluant la colonne protégée)
quasi_constantes = [
    col for col in data.columns
    if (col != colonne_protegee) and  # Exclusion explicite
       (data[col].value_counts(normalize=True).max() > seuil)
]


In [51]:
# 2. Affichage des résultats
print(f"Colonnes quasi-constantes (≥{seuil:.0%} de valeurs identiques):")
print(pd.DataFrame({
    'Colonne': quasi_constantes,
    'Valeur dominante': [data[col].mode()[0] for col in quasi_constantes],
    'Fréquence': [data[col].value_counts(normalize=True).max() for col in quasi_constantes]
}))

Colonnes quasi-constantes (≥90% de valeurs identiques):
                   Colonne Valeur dominante  Fréquence
0                Source IP       172.16.0.5   0.998508
1           Destination IP     192.168.50.1   0.998510
2                 Protocol               17   0.961944
3   Init_Win_bytes_forward               -1   0.961996
4  Init_Win_bytes_backward               -1   0.993542
5                  Inbound                1   0.998708


In [52]:
# 3. Suppression (en conservant la colonne protégée)
data = data.drop(columns=quasi_constantes)
# 4. Vérification
print(f"\nDimensions: {data.shape}")
# Afficher les valeurs uniques et leur nombre
print(data["Label"].value_counts())


Dimensions: (48700314, 31)
Label
TFTP             19515971
DrDoS_SNMP        5149261
DrDoS_DNS         4908665
DrDoS_MSSQL       4396046
DrDoS_NetBIOS     3963446
DrDoS_UDP         3094002
DrDoS_SSDP        2568569
DrDoS_LDAP        2141300
Syn               1379983
DrDoS_NTP         1195690
UDP-lag            330079
BENIGN              56863
WebDDoS               439
Name: count, dtype: int64


In [53]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 48700314 entries, 0 to 50063111
Data columns (total 31 columns):
 #   Column                       Dtype  
---  ------                       -----  
 0   Unnamed: 0                   int64  
 1   Flow ID                      object 
 2   Source Port                  int64  
 3   Destination Port             int64  
 4   Timestamp                    object 
 5   Flow Duration                int64  
 6   Total Fwd Packets            int64  
 7   Total Length of Fwd Packets  float64
 8   Fwd Packet Length Max        float64
 9   Fwd Packet Length Min        float64
 10  Fwd Packet Length Mean       float64
 11  Flow IAT Mean                float64
 12  Flow IAT Std                 float64
 13  Flow IAT Max                 float64
 14  Flow IAT Min                 float64
 15  Fwd IAT Total                float64
 16  Fwd IAT Mean                 float64
 17  Fwd IAT Std                  float64
 18  Fwd IAT Max                  float64
 19  Fwd

# Métadonnées techniques

In [54]:
#Suppression
data.drop(columns=['Unnamed: 0', 'Flow ID', 'Timestamp'], inplace=True)
data.shape
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 48700314 entries, 0 to 50063111
Data columns (total 28 columns):
 #   Column                       Dtype  
---  ------                       -----  
 0   Source Port                  int64  
 1   Destination Port             int64  
 2   Flow Duration                int64  
 3   Total Fwd Packets            int64  
 4   Total Length of Fwd Packets  float64
 5   Fwd Packet Length Max        float64
 6   Fwd Packet Length Min        float64
 7   Fwd Packet Length Mean       float64
 8   Flow IAT Mean                float64
 9   Flow IAT Std                 float64
 10  Flow IAT Max                 float64
 11  Flow IAT Min                 float64
 12  Fwd IAT Total                float64
 13  Fwd IAT Mean                 float64
 14  Fwd IAT Std                  float64
 15  Fwd IAT Max                  float64
 16  Fwd IAT Min                  float64
 17  Fwd Header Length            int64  
 18  Fwd Packets/s                float64
 19  Min

# Encodage

In [55]:
# Label Encoding (pour la classification)
from sklearn.preprocessing import LabelEncoder
data['Label'] = LabelEncoder().fit_transform(data['Label'])


In [ ]:
# Sauvegarder le DataFrame nettoyé localement
data.to_csv('/content/ddos_nettoye.csv', index=False)


In [ ]:
# Importer la bibliothèque de Google Cloud Storage
from google.cloud import storage

# Créer un client
client = storage.Client()

# Paramètres
bucket_name = 'mon-bucket-22go'  # Remplace si nécessaire
destination_blob_name = 'ddos_nettoye.csv'
source_file_name = '/content/ddos_nettoye.csv'

# Uploader vers le bucket
bucket = client.bucket(bucket_name)
blob = bucket.blob(destination_blob_name)
blob.upload_from_filename(source_file_name)

print("✅ Fichier ddos_nettoye.csv téléversé dans le bucket !")
